In [ ]:
import sys
# For notebooks inside research/ftir_hips_chem/:
sys.path.insert(0, './scripts')
# For notebooks under notebooks/ (e.g. plotting_gaps_scenarios.ipynb):
# sys.path.insert(0, '../research/ftir_hips_chem/scripts')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Config + data
from config import (
    SITES, PROCESSED_SITES_DIR, FILTER_DATA_PATH,
    AERONET_DATA_DIR, WEATHER_DATA_DIR, MAC_VALUE,
)

# Exclusions (use these — do not hand-filter)
from outliers import (
    EXCLUDED_SAMPLES, MANUAL_OUTLIERS,
    apply_exclusion_flags, apply_threshold_flags,
    get_clean_data, print_exclusion_summary,
)

# Data loading / matching
from data_matching import (
    load_aethalometer_data, load_filter_data,
    match_aeth_filter_data, match_all_parameters,
)
from etad_factors import load_etad_factor_contributions, match_etad_factors

# External datasets — these resolve their own location, so do NOT hardcode a
# Drive path. Each checks an env var, then a config constant, then discovers the
# Drive mount: AETHMODULAR_AERONET_DIR / AETHMODULAR_IMPROVE_DIR.
from aeronet import load_aeronet, aeronet_dir, COLS as AERONET_COLS
from improve_io import load_improve_clean

# Plotting — importing the package auto-applies the white-background default
# style (apply_default_style()). Do NOT call plt.style.use('seaborn-v0_8-darkgrid')
# afterwards — it re-adds the grey axes facecolor we don't want.
from plotting import PlotConfig, crossplots, timeseries, distributions, comparisons
from plotting.utils import calculate_regression_stats

PlotConfig.set(sites='all', layout='individual', show_stats=False, show_1to1=False)


# Filter-only scientific diagnostics

Six reproducible figure families. The workflow verifies frozen input hashes, reapplies the canonical exclusion registry to original replicate IDs, and uses `get_clean_data` before diagnostic statistics. The 480 ratio points must be a subset of the 545 diagnostic points. ChemSpec and instrument timing gaps do not alter these populations. No uncertainty-weighted or independent EC calibration fit is performed.

In [ ]:
from pathlib import Path
from IPython.display import display, Image
sys.path.insert(0, './workflows')
from analyze_filter_diagnostics import main
from plotting.filter_diagnostics import FIGURES
TABLES = main()
PLOTS = Path('output/plots/filter_diagnostics')
points = pd.read_parquet(TABLES / 'analysis_points.parquet')
display(pd.read_parquet(TABLES / 'site_results.parquet'))

## Within-site HIPS versus FTIR-predicted EC

The diagnostic population contains 545 eligible physical-filter pairs. Open rings mark the 65 predictions that remain diagnostic but cannot be divided into HIPS under the baseline rule. Fits are descriptive unweighted OLS with an intercept; their R² values do not establish independent EC validation. Site-specific axes retain negative predictions. Both variables have measurement error, so slopes should not be read as unbiased calibration coefficients.

In [ ]:
display(Image(filename=str(PLOTS / '01_site_relationships.png')))
# Regenerated by main() using the corresponding function in plotting.filter_diagnostics.

## Ratio distributions

All 480 eligible ratios are shown, with medians and interquartile boxes. Ratios use the original HIPS value in Mm⁻¹ divided by the original FTIR-predicted EC in µg m⁻³. No denominator is substituted. The site distributions overlap substantially; differences in their medians are descriptive and do not isolate source, seasonal or instrument effects. All individual points, including the large Addis ratio, remain visible.

In [ ]:
display(Image(filename=str(PLOTS / '02_ratio_distributions.png')))
# Regenerated by main() using the corresponding function in plotting.filter_diagnostics.

## Denominator behavior

Open rings identify EC from 1 to less than 2 times its own reported MDL. This annotation is not an added exclusion. An inverse relationship with the denominator can arise algebraically; no regression or causal interpretation is applied to this panel. The report and denominator-strata table give counts and medians near and above that range.

In [ ]:
display(Image(filename=str(PLOTS / '03_ratio_denominators.png')))
# Regenerated by main() using the corresponding function in plotting.filter_diagnostics.

## Reported-date patterns

These are reported filter dates, not verified active periods. No observations are interpolated or connected through gaps. Dashed lines are overall site medians. The graph suggests date structure, especially at Addis, but the analysis does not infer a mechanism or fit a temporal trend. Full date ranges and point links are retained in the tables.

In [ ]:
display(Image(filename=str(PLOTS / '04_ratio_dates.png')))
# Regenerated by main() using the corresponding function in plotting.filter_diagnostics.

## Sensitivity to stricter denominators

The declared descriptive grid is 1, 1.5, 2, 3 and 5 times the filter-specific MDL. The original 480-pair eligibility is unchanged. Each stricter scenario reports its own count and median, with every selected filter linked in sensitivity_point_links.parquet. A missing median means no retained points, not a zero ratio. No threshold is optimized for measurement agreement.

In [ ]:
display(Image(filename=str(PLOTS / '05_denominator_sensitivity.png')))
# Regenerated by main() using the corresponding function in plotting.filter_diagnostics.

## Registered exclusion

The original Delhi extreme point remains visible in red and traceable to INDH-0172. Its registered exclusion predates this report. The fit uses only the 62 eligible Delhi diagnostic pairs and ends within their EC range. This audit view does not reintroduce the point into the 545-pair population.

In [ ]:
display(Image(filename=str(PLOTS / '06_registry_context.png')))
# Regenerated by main() using the corresponding function in plotting.filter_diagnostics.

## Traceability and retrieval priorities

The queue uses reported envelope overlap for retrieval only. The earlier ChemSpec CSVs already contain the conflicting EC values; the recovered historical importer copies Value and MDL separately. Run logs and scoped instrument history are still required for a real interval subset.

In [ ]:
queue = pd.read_parquet(TABLES / 'evidence_recovery_queue.parquet')
display(queue.loc[queue.priority <= 2, ['site', 'base_filter_id', 'reported_start_utc', 'missing_evidence']].groupby('site').head(4))
display(pd.read_parquet(TABLES / 'ratio_exclusion_reasons.parquet')[['site','reason','count']])
print(TABLES / 'results_report.md')